# 01 - Dataset audit and manifest
Reports what is **actually** in the attached MetaPKLot dataset: file counts, image sizes, corrupt files, duplicates, annotation files and their structure, and hints about cameras/dates/sequences. It assumes no layout and invents nothing.

**Inputs:** the MetaPKLot dataset attached via *Add Input*. **Outputs (in `/kaggle/working/artifacts/audit/`):** `audit_summary.json`, `manifests/image_inventory.csv`, `manifests/annotation_inventory.csv`, `manifests/dataset_hashes.json`, `corrupt_report.json`, `duplicate_report.json`, `tree.txt`, `status.json`.

**Run order:** first with `SMOKE = True` (fast sample), check the output, then `SMOKE = False` for the full audit.

In [ ]:
# --- Setup: get the project code onto Kaggle ---
# Preferred: attach the code as a Kaggle dataset (works for a private GitHub repo).
# Fallback: git clone (only works if the GitHub repo is public and Internet is ON).
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/Amith2277/Edge-Computing-EndSem.git"
REPO_DIR = "/kaggle/working/repo"
MARK = os.path.join("src", "common", "config.py")

def find_attached_code():
    for pat in ("/kaggle/input/*/", "/kaggle/input/*/*/", "/kaggle/input/*/*/*/"):
        for d in sorted(glob.glob(pat)):
            if os.path.isfile(os.path.join(d, MARK)):
                return d.rstrip("/")
    return None

if os.path.isdir(REPO_DIR):
    print("using existing", REPO_DIR)
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):  # cloned earlier: fetch the latest code
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout.strip())
else:
    attached = find_attached_code()
    if attached:
        shutil.copytree(attached, REPO_DIR)
        print("copied attached code dataset:", attached)
    else:
        env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, universal_newlines=True, env=env)
        print(r.stdout)
        if r.returncode != 0:
            raise SystemExit("No attached code dataset found and git clone failed (private repo?). "
                             "Upload adaptivepark-code.zip as a Kaggle dataset and Add Input it, then re-run.")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("code at", REPO_DIR, "| files:", sorted(os.listdir(REPO_DIR))[:12])


In [ ]:
# Step 1: what is attached? (deeper than the top level, so wrapper folders are visible)
import os
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root[len("/kaggle/input"):].count(os.sep)
    if depth <= 4:
        print("  " * depth + os.path.basename(root) + "/", "[{} files, {} dirs]".format(len(files), len(dirs)))
    if depth >= 4:
        dirs[:] = []

In [ ]:
# Step 2: configuration. The dataset folder is detected automatically.
SMOKE = True            # True = audit an evenly spaced sample; set False for the full run
DATASET_HINT = ""       # only needed if several datasets are attached: part of the right path, e.g. "metapklot"
DATA_ROOT = None        # or set the path yourself to skip detection
OUT = "/kaggle/working/artifacts/audit"

from src.data.discover import describe, find_dataset_root
if DATA_ROOT is None:
    DATA_ROOT = str(find_dataset_root("/kaggle/input", DATASET_HINT))
info = describe(DATA_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("files:", info["files"], "| top-level entries:", info["n_children"], info["children"])

In [ ]:
# Step 3: run the audit.
import subprocess, sys
cmd = [sys.executable, "scripts/audit_dataset.py", "--root", DATA_ROOT, "--out", OUT] + (["--smoke"] if SMOKE else [])
r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
print(r.stdout[-6000:])
assert r.returncode == 0, "audit failed, see output above"

In [ ]:
# Step 4: folder tree and annotation structure (these decide what we can evaluate).
print(open(OUT + "/tree.txt").read()[:4000])
import pandas as pd
a = pd.read_csv(OUT + "/manifests/annotation_inventory.csv")
pd.set_option("display.max_colwidth", 140, "display.width", 200)
print(a[["path", "kind", "detail"]].head(30).to_string())

## What to send back
Paste the printed `audit_summary.json`, the folder tree, and the annotation table. From them we decide: are there vehicle boxes or masks, slot polygons, occupancy labels, sequences/timestamps, and how to split.

If **no slot polygons or occupancy labels** are found, slot-level evaluation is marked **BLOCKED** (we never invent polygons).

Then set `SMOKE = False`, run again, and **Save Version** so the manifests can be attached to the next notebooks.